# Tools and structured output — OpenAI GPT on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Client-side tool calling and strict JSON on the Responses API. This is the
machinery behind most agent loops: define tools, let the model choose, execute,
feed results back, repeat.

**Models used here:** `openai.gpt-5.6-sol` (frontier, `/openai/v1`) and
`openai.gpt-oss-120b` (open-weight, bare `/v1`). §5b sends the same request to
seven of the GPT models this folder covers, so it also touches the GPT-6 and
`gpt-oss-20b` IDs.

## What this notebook covers
- The Responses tool shape and the full call/result loop
- `tool_choice`: auto, required, and forcing a specific function
- Parallel tool calls
- Strict JSON two ways — native `text.format` and the forced-tool trick, and
  which models honour each
- Robust parsing, and the failure modes that actually happen in production

## Self-contained, but see also
- **Core Responses API for this family** → `01-responses-api-core.ipynb`
- **Server-side tools (Lambda MCP (Model Context Protocol), notes/tasks)** →
  `05-server-side-tools-and-fine-tuning.ipynb`
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `response_text` | assistant text from a Responses API payload |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys

sys.path.insert(0, "../_shared")
from bedrock import err, parse_json_lenient, post, response_text

REGION = "us-east-1"
SOL = "openai.gpt-5.6-sol"
OSS120 = "openai.gpt-oss-120b"

GPT5_PREFIX = "/openai/v1"  # the hosted GPT line, gpt-6 included
OSS_PREFIX = "/v1"  # gpt-oss

from aws_bedrock_token_generator import provide_token
from openai import OpenAI

client = OpenAI(
    api_key=provide_token(region=REGION),
    base_url=f"https://bedrock-mantle.{REGION}.api.aws{GPT5_PREFIX}",
)
print("endpoint:", f"https://bedrock-mantle.{REGION}.api.aws{GPT5_PREFIX}")

endpoint: https://bedrock-mantle.us-east-1.api.aws/openai/v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


openai.gpt-5.6-sol                     us.openai.gpt-5.6-sol                    mantle, runtime


openai.gpt-oss-120b                    openai.gpt-oss-120b-1:0                  mantle, runtime



=> 2/2 of these are on bedrock-runtime; 2 under a different id.
   Every model here is on both endpoints. This notebook shows the
   bedrock-mantle calls; the ids above are what you send to switch.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. The Responses tool shape

`name`, `description` and `parameters` sit at the **top level** of the tool
object. Chat Completions nests them under `"function"` — a common source of
confusion when porting code between the two APIs.

In [3]:
RESPONSES_SHAPE = {
    "type": "function",
    "name": "get_stock",  # top level
    "description": "Look up stock for a SKU.",
    "parameters": {
        "type": "object",
        "properties": {"sku": {"type": "string"}},
        "required": ["sku"],
    },
}

CHAT_COMPLETIONS_SHAPE = {
    "type": "function",
    "function": {  # nested
        "name": "get_stock",
        "description": "Look up stock for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string"}},
            "required": ["sku"],
        },
    },
}

print("Responses API tool:")
print(json.dumps(RESPONSES_SHAPE, indent=2))
print("\nChat Completions tool (same tool, different nesting):")
print(json.dumps(CHAT_COMPLETIONS_SHAPE, indent=2))

Responses API tool:
{
  "type": "function",
  "name": "get_stock",
  "description": "Look up stock for a SKU.",
  "parameters": {
    "type": "object",
    "properties": {
      "sku": {
        "type": "string"
      }
    },
    "required": [
      "sku"
    ]
  }
}

Chat Completions tool (same tool, different nesting):
{
  "type": "function",
  "function": {
    "name": "get_stock",
    "description": "Look up stock for a SKU.",
    "parameters": {
      "type": "object",
      "properties": {
        "sku": {
          "type": "string"
        }
      },
      "required": [
        "sku"
      ]
    }
  }
}


## 2. A complete tool loop

Four steps: send tools → model requests a call → you execute → send the result
back. The `call_id` is what ties a result to its request.

In [4]:
INVENTORY = {"A-100": 42, "B-200": 0, "C-300": 7}
SHIPPING = {"standard": 4.99, "express": 12.50}


def get_stock(sku: str) -> dict:
    qty = INVENTORY.get(sku.upper(), 0)
    return {"sku": sku.upper(), "quantity": qty, "in_stock": qty > 0}


def get_shipping(method: str) -> dict:
    return {"method": method, "cost_usd": SHIPPING.get(method.lower())}


TOOLS = [
    {
        "type": "function",
        "name": "get_stock",
        "description": "Look up stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {
                "sku": {"type": "string", "description": "SKU code, e.g. A-100"}
            },
            "required": ["sku"],
        },
    },
    {
        "type": "function",
        "name": "get_shipping",
        "description": "Look up the cost of a shipping method.",
        "parameters": {
            "type": "object",
            "properties": {
                "method": {"type": "string", "enum": ["standard", "express"]}
            },
            "required": ["method"],
        },
    },
]

DISPATCH = {"get_stock": get_stock, "get_shipping": get_shipping}

The loop. `max_rounds` bounds it: an agent with no bound can spin forever on a
task it cannot finish (OWASP LLM10, Unbounded Consumption).

In [5]:
def run_tool_loop(
    question: str, model: str = SOL, max_rounds: int = 5, prefix: str = GPT5_PREFIX
) -> dict:
    """Full client-side tool loop. Returns the answer and a trace of calls."""
    conversation = [{"role": "user", "content": question}]
    trace = []
    for round_no in range(max_rounds):
        code, data = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": conversation,
                "tools": TOOLS,
                "tool_choice": "auto",
                "max_output_tokens": 600,
                "store": False,
            },
            region=REGION,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")

        calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
        if not calls:
            return {
                "answer": response_text(data),
                "trace": trace,
                "rounds": round_no + 1,
            }

        for call in calls:
            # parse_json_lenient rather than json.loads: arguments occasionally
            # carry trailing characters after a valid object.
            args = parse_json_lenient(call["arguments"] or "{}")
            fn = DISPATCH.get(call["name"])
            result = fn(**args) if fn else {"error": f"unknown tool {call['name']}"}
            trace.append(
                {
                    "round": round_no + 1,
                    "tool": call["name"],
                    "args": args,
                    "result": result,
                }
            )
            # Echo the call, then its result. call_id links the two.
            conversation.append(
                {
                    "type": "function_call",
                    "call_id": call["call_id"],
                    "name": call["name"],
                    "arguments": call["arguments"],
                }
            )
            conversation.append(
                {
                    "type": "function_call_output",
                    "call_id": call["call_id"],
                    "output": json.dumps(result),
                }
            )
    return {"answer": "(max rounds reached)", "trace": trace, "rounds": max_rounds}


result = run_tool_loop("Is SKU A-100 in stock, and what does express shipping cost?")

Run it and inspect the trace.

In [6]:
print("rounds:", result["rounds"])
for step in result["trace"]:
    print(
        f"  round {step['round']}: {step['tool']}({step['args']}) -> {step['result']}"
    )
print("\nanswer:", result["answer"][:300])

rounds: 2
  round 1: get_stock({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42, 'in_stock': True}
  round 1: get_shipping({'method': 'express'}) -> {'method': 'express', 'cost_usd': 12.5}

answer: Yes. SKU A-100 is in stock with **42 units available**. Express shipping costs **$12.50**.


**Note what we did not append:** reasoning items. Send tool calls and their
outputs, plus final answers — never the model's own reasoning traces. Replaying
reasoning degrades later turns.

## 3. `tool_choice`

Three modes, in increasing order of compulsion.

In [7]:
QUESTION = "Do we have C-300 in stock?"
print(f"{'model':22} {'tool_choice':30} {'HTTP':>6} {'calls':>6}")
print("-" * 70)
for model, prefix in ((SOL, GPT5_PREFIX), (OSS120, OSS_PREFIX)):
    for choice in (
        "auto",
        "none",
        "required",
        {"type": "function", "name": "get_stock"},
    ):
        code, data = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": QUESTION,
                "tools": TOOLS,
                "tool_choice": choice,
                "max_output_tokens": 400,
                "store": False,
            },
            region=REGION,
        )
        calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
        label = choice if isinstance(choice, str) else "{'type':'function',...}"
        print(f"{model:22} {label:30} {code:>6} {len(calls):>6}")
        if code != 200:
            print(f"      {err(data)[:96]}")

model                  tool_choice                      HTTP  calls
----------------------------------------------------------------------


openai.gpt-5.6-sol     auto                              200      1


openai.gpt-5.6-sol     none                              200      0


openai.gpt-5.6-sol     required                          200      1


openai.gpt-5.6-sol     {'type':'function',...}           200      1


openai.gpt-oss-120b    auto                              200      1


openai.gpt-oss-120b    none                              400      0
      The model 'openai.gpt-oss-120b' does not support tool_choice '"none"' for /v1/responses API. Sup


openai.gpt-oss-120b    required                          400      0
      The model 'openai.gpt-oss-120b' does not support tool_choice '"required"' for /v1/responses API.


openai.gpt-oss-120b    {'type':'function',...}           200      0


### `tool_choice` support is per model — and this one bites

- Some models accept every form: `auto`, `none`, `required`, and a named function.
- Others accept only a subset. The cell below probes each form, because which
  model supports what changes. In the run committed here, `none` and `required` were
  explicit 400s ("Supported options: [\"auto\"]"), and the *named-function* form
  returned **HTTP 200 while quietly ignoring the constraint** — the model answered
  in prose with no tool call at all.

That last case is the dangerous one: a 200 that silently did not do what you
asked. Never assume a forced tool call succeeded; always check for the call.

In [8]:
# Forcing one specific function by name.
code, data = post(
    f"{GPT5_PREFIX}/responses",
    {
        "model": SOL,
        "input": "What about shipping?",
        "tools": TOOLS,
        "tool_choice": {"type": "function", "name": "get_shipping"},
        "max_output_tokens": 400,
        "store": False,
    },
    region=REGION,
)
calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
print(f"forced get_shipping -> HTTP {code} | calls={[c['name'] for c in calls]}")
if calls:
    print("arguments:", calls[0]["arguments"])

forced get_shipping -> HTTP 200 | calls=['get_shipping']
arguments: {"method":"standard"}


## 4. Parallel tool calls

Unlike some families (Gemma 4 issues one call per turn), gpt-5.6 accepts
`parallel_tool_calls` and can request several at once.

In [9]:
code, data = post(
    f"{GPT5_PREFIX}/responses",
    {
        "model": SOL,
        "input": (
            "Check stock for A-100 and B-200, and get express "
            "shipping cost. All of it."
        ),
        "tools": TOOLS,
        "parallel_tool_calls": True,
        "max_output_tokens": 600,
        "store": False,
    },
    region=REGION,
)
calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
print(f"HTTP {code} | {len(calls)} call(s) requested in one turn:")
for call in calls:
    print(f"   {call['name']}({call['arguments']})")

HTTP 200 | 3 call(s) requested in one turn:
   get_stock({"sku":"A-100"})
   get_stock({"sku":"B-200"})
   get_shipping({"method":"express"})


In [10]:
# max_tool_calls is DOCUMENTED as a cap on how many calls the model may make. Send
# it and count what comes back, because accepted is not the same as honoured.
print(f"{'max_tool_calls':>15} {'HTTP':>5} {'calls requested':>16}")
print("-" * 40)
observed = {}
for cap in (1, 2, None):
    body = {
        "model": SOL,
        "input": "Check stock for A-100, B-200 and C-300, plus both shipping methods.",
        "tools": TOOLS,
        "max_output_tokens": 600,
        "store": False,
    }
    if cap is not None:
        body["max_tool_calls"] = cap
    code, data = post(f"{GPT5_PREFIX}/responses", body, region=REGION)
    calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
    observed[cap] = len(calls)
    print(f"{cap!s:>15} {code:>5} {len(calls):>16}")

print()
if observed.get(1, 0) > 1 or observed.get(2, 0) > 2:
    print("=> The cap was ACCEPTED (HTTP 200) and IGNORED. This is the '200 does not")
    print("   mean honoured' failure mode, and it is the dangerous kind: there is no")
    print("   error to alert you. If you need a hard limit on tool calls, enforce it")
    print("   in your own loop -- count the calls you execute and stop.")
else:
    print("=> The cap was honoured on this run. Verify it yourself rather than")
    print("   relying on it: enforce the limit in your loop as well.")

 max_tool_calls  HTTP  calls requested
----------------------------------------


              1   200                5


              2   200                5


           None   200                5

=> The cap was ACCEPTED (HTTP 200) and IGNORED. This is the '200 does not
   mean honoured' failure mode, and it is the dangerous kind: there is no
   error to alert you. If you need a hard limit on tool calls, enforce it
   in your own loop -- count the calls you execute and stop.


## 5. Strict JSON via `text.format`

The native route: give a JSON Schema and set `strict: True`.

In [11]:
REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "topics": {"type": "array", "items": {"type": "string"}},
        "would_recommend": {"type": "boolean"},
    },
    "required": ["summary", "sentiment", "topics", "would_recommend"],
    "additionalProperties": False,
}

REVIEW = (
    "Battery life is superb and it charges fast, but the screen scratches "
    "far too easily and support took a week to reply."
)

code, data = post(
    f"{GPT5_PREFIX}/responses",
    {
        "model": SOL,
        "input": f"Analyse this review:\n{REVIEW}",
        "max_output_tokens": 700,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "review",
                "schema": REVIEW_SCHEMA,
                "strict": True,
            }
        },
        "store": False,
    },
    region=REGION,
)
raw = response_text(data)
print("HTTP", code, "| raw:", repr(raw[:120]))
parsed = parse_json_lenient(raw)
print(json.dumps(parsed, indent=2))
expected = set(REVIEW_SCHEMA["properties"])
if set(parsed) != expected:
    raise ValueError(
        f"schema mismatch: expected {sorted(expected)}, got {sorted(parsed)}"
    )
print("schema honoured exactly:", sorted(parsed))

HTTP 200 | raw: '{"summary":"The product has excellent battery life and fast charging, but its screen scratches too easily and customer s'
{
  "summary": "The product has excellent battery life and fast charging, but its screen scratches too easily and customer support is slow to respond.",
  "sentiment": "neutral",
  "topics": [
    "battery life",
    "charging speed",
    "screen durability",
    "customer support"
  ],
  "would_recommend": false
}
schema honoured exactly: ['sentiment', 'summary', 'topics', 'would_recommend']


### 5b. Which models honour it, and the route that returns nothing

Section 5 sent that request to one model. Two things it leaves open for anyone
choosing a model: whether the GPT-6 line now in the catalogue accepts the same
shape, and what the open-weight models covered in §7 do with it.

Each route is sampled twice, because one call cannot separate a behaviour from a
coin flip, and each verdict is computed from the response body rather than from the
status code. Where a route returns nothing the cell re-sends the same request with
the `text` block removed: without that control, "the field causes it" would be an
inference about a route that is also capable of being slow.

In [12]:
# Which models honour `text.format`, measured per (model, route). The summary table
# in section 6 used to key this on "gpt-5.x" -- a model GENERATION -- and the GPT-6
# line is now in the catalogue, so the models are enumerated here instead. Keying a
# rule on a version number has been wrong twice in this collection already; see the
# _OPENAI_HOSTED_GPT comment in ../_shared/bedrock.py.
#
# post() defaults to 5 attempts and a 240-second socket timeout. One route below does
# not answer at all, and those defaults would make that row a 20-minute wait, so both
# are pinned here. status -1 from post() means "nothing came back within TIMEOUT",
# which is a different result from a service error and is reported as one.
import time

OSS20 = "openai.gpt-oss-20b"
TRIES = 2  # one call cannot tell a behaviour from a coin flip
TIMEOUT = 45

ROUTES = [
    ("openai.gpt-6-sol", GPT5_PREFIX),
    ("openai.gpt-6-luna", GPT5_PREFIX),
    (SOL, GPT5_PREFIX),
    ("openai.gpt-5.5", GPT5_PREFIX),
    ("openai.gpt-5.4", GPT5_PREFIX),
    (OSS120, OSS_PREFIX),
    (OSS20, OSS_PREFIX),
]
FORMAT = {
    "format": {
        "type": "json_schema",
        "name": "review",
        "schema": REVIEW_SCHEMA,
        "strict": True,
    }
}


def responses_call(model: str, prefix: str, *, constrain: bool):
    """One /responses call, timed. `constrain` decides whether `text` is sent at all."""
    body = {
        "model": model,
        "input": f"Analyse this review:\n{REVIEW}",
        "max_output_tokens": 2000,
        "store": False,
    }
    if constrain:
        body["text"] = FORMAT
    started = time.perf_counter()
    code, data = post(
        f"{prefix}/responses", body, region=REGION, attempts=1, timeout=TIMEOUT
    )
    return code, data, time.perf_counter() - started


def verdict_for(code: int, data: dict) -> tuple[str, str]:
    """Read a verdict off one response. Nothing here is decided by the status alone."""
    if code == -1:
        return "no answer", f"nothing within {TIMEOUT}s"
    if code != 200:
        return f"HTTP {code}", err(data)[:58]
    raw = response_text(data)
    # An empty string here is the token cap, not a missing feature. Say which.
    if not raw.strip():
        return "empty", f"status={data.get('status')} {data.get('incomplete_details')}"
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return "not JSON", repr(raw[:36])
    if set(parsed) != set(REVIEW_SCHEMA["properties"]):
        return "wrong keys", repr(sorted(parsed))
    return "honoured", f"sentiment={parsed['sentiment']}"


print(f"{'model':22} {'route':22} {'verdict(s)':24} detail")
print("-" * 96)
verdicts = {}
for model, prefix in ROUTES:
    seen = [verdict_for(*responses_call(model, prefix, constrain=True)[:2])
            for _ in range(TRIES)]
    labels = [v for v, _ in seen]
    verdicts[model] = set(labels)
    shown = " / ".join(labels) if len(set(labels)) > 1 else labels[0]
    print(f"{model:22} {prefix + '/responses':22} {shown:24} {seen[-1][1][:30]}")

split = {m: v for m, v in verdicts.items() if len(v) > 1}
honoured = [m for m, v in verdicts.items() if v == {"honoured"}]
silent = [m for m, v in verdicts.items() if v == {"no answer"}]
print()
if split:
    print(f"=> {len(split)} model(s) gave different verdicts across {TRIES} identical")
    print("   calls, so the rows above are not yet properties of the model:")
    for model, vs in split.items():
        print(f"     {model:22} {sorted(vs)}")
print(f"=> honoured by {len(honoured)}/{len(ROUTES)}: {', '.join(honoured) or 'none'}")
if honoured:
    g6 = [m for m in honoured if m.startswith("openai.gpt-6")]
    print(f"   {len(g6)} of them are GPT-6 ({', '.join(g6) or 'none'}), so an")
    print("   availability rule keyed on 'gpt-5.x' excludes models that support it.")

# The two silent rows are the interesting ones, and "the field causes it" would be an
# inference rather than a measurement without this: the same models, the same path and
# the same timeout, with the text block removed and nothing else changed.
if silent:
    print(f"\n=> NO ANSWER from {len(silent)}: {', '.join(silent)}, {TRIES}/{TRIES} calls")
    print(f"   each, nothing within {TIMEOUT}s. Control, same route without `text`:")
    control = {}
    for model in silent:
        for _ in range(TRIES):
            code, data, secs = responses_call(model, OSS_PREFIX, constrain=False)
            control.setdefault(model, []).append(code)
            answer = response_text(data)[:24] if code == 200 else err(data)[:24]
            print(f"     {model:22} HTTP {code:>4} in {secs:5.1f}s  {answer!r}")
    flat = [c for codes in control.values() for c in codes]
    print(f"   control answered {sum(1 for c in flat if c == 200)}/{len(flat)},"
          f" timed out {sum(1 for c in flat if c == -1)}/{len(flat)}")
    print(f"=> so it is the `text` block and not the route: {all(c == 200 for c in flat)}")
    print("   A 400 naming an unsupported field is something a caller can handle.")
    print("   This is not that: the request does not come back, so a caller with no")
    print("   client-side timeout of its own has nothing to stop it waiting. The")
    print("   control is sampled rather than run once because this route can also")
    print("   time out with no `text` block in it.")

model                  route                  verdict(s)               detail
------------------------------------------------------------------------------------------------
openai.gpt-6-sol       /openai/v1/responses   honoured                 sentiment=negative
openai.gpt-6-luna      /openai/v1/responses   honoured                 sentiment=negative
openai.gpt-5.6-sol     /openai/v1/responses   honoured                 sentiment=neutral
openai.gpt-5.5         /openai/v1/responses   honoured                 sentiment=neutral
openai.gpt-5.4         /openai/v1/responses   honoured                 sentiment=neutral
openai.gpt-oss-120b    /v1/responses          no answer                nothing within 45s
openai.gpt-oss-20b     /v1/responses          no answer                nothing within 45s

=> honoured by 5/7: openai.gpt-6-sol, openai.gpt-6-luna, openai.gpt-5.6-sol, openai.gpt-5.5, openai.gpt-5.4
   2 of them are GPT-6 (openai.gpt-6-sol, openai.gpt-6-luna), so an
   availability rule 

Those last two rows are not a refusal, and the control underneath them rules out the
route: the same models on the same path answer once `text` is gone. A 400 naming an
unsupported field is something a caller can handle. A request that does not come
back is not, so anything sending `text.format` to `gpt-oss` needs a client-side
timeout of its own.

What should a `gpt-oss` caller send instead, and what does the other endpoint do with
the request that stalls here?

In [13]:
# So what does a gpt-oss caller do instead? Two things the cell above does not settle:
# whether Chat Completions accepts the same schema, and what the OTHER endpoint does
# with the request that hangs here.
from bedrock import runtime_post

print(f"{'model':22} {'HTTP':>5} {'json.loads':>11} {'lenient':>8} {'keys':>6}  content[:34]")
print("-" * 96)
cc = {}
for model in (OSS120, OSS20):
    for _ in range(TRIES):
        code, data = post(
            f"{OSS_PREFIX}/chat/completions",
            {
                "model": model,
                "messages": [
                    {"role": "user", "content": f"Analyse this review:\n{REVIEW}"}
                ],
                "max_completion_tokens": 2000,
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": "review",
                        "schema": REVIEW_SCHEMA,
                        "strict": True,
                    },
                },
            },
            region=REGION,
            attempts=1,
            timeout=TIMEOUT,
        )
        choices = data.get("choices") or [{}]
        content = (choices[0].get("message") or {}).get("content") or ""
        try:
            json.loads(content)
            strict_load = "ok"
        except (json.JSONDecodeError, TypeError):
            strict_load = "raises"
        try:
            keys = sorted(parse_json_lenient(content))
            lenient = "ok"
        except (ValueError, TypeError):
            keys, lenient = [], "raises"
        matched = keys == sorted(REVIEW_SCHEMA["properties"])
        cc.setdefault(model, []).append((code, strict_load, lenient, matched))
        print(
            f"{model:22} {code:>5} {strict_load:>11} {lenient:>8} "
            f"{str(matched):>6}  {content[:34]!r}"
        )

runs = [r for rows in cc.values() for r in rows]
answered = [r for r in runs if r[0] == 200]
print()
print(f"=> answered: {len(answered)}/{len(runs)}"
      f" (status -1 is this route not coming back within {TIMEOUT}s,"
      " the same result as the table above)")
print(f"=> of those, json.loads() raised on"
      f" {sum(1 for _, s, *_ in answered if s == 'raises')}/{len(answered)}"
      f" and parse_json_lenient() recovered"
      f" {sum(1 for _, _, l, _ in answered if l == 'ok')}/{len(answered)}")
print(f"=> schema keys exactly right after the lenient parse:"
      f" {sum(1 for *_, m in answered if m)}/{len(answered)}")
# Do not harden any of these counts into prose. Across five runs while this cell was
# written: one call returned a wrapper parse_json_lenient() could not recover at all
# (lenient "raises"), another run recovered an object whose keys did NOT match the
# schema (lenient ok, keys False), and one call did not come back at all. That is why
# the keys column is computed separately from the parse, and why the lines below name
# the mechanism rather than today's numbers.
print("   So response_format here is accepted and loosely followed, not enforced.")
print("   The schema's fields come back. Whether they arrive as a bare JSON object")
print("   or wrapped in a prose heading that json.loads() rejects varies per call,")
print("   so read the counts above rather than the status: 200 is not the same as")
print("   honoured, the counts move between runs of this same cell, and a lenient")
print("   parse that succeeds can still hand back the wrong keys, which is why")
print("   that is a column of its own. Validate the parsed object against your")
print("   schema before anything downstream sees it.")

# The same request that does not come back on mantle, sent to bedrock-runtime.
# Timed, because "it fails fast here" is otherwise a claim and not a measurement.
runtime_id = runtime_id_for(OSS120, REGION)
started = time.perf_counter()
code, data = runtime_post(
    "/openai/v1/responses",
    {
        "model": runtime_id,
        "input": f"Analyse this review:\n{REVIEW}",
        "max_output_tokens": 2000,
        "text": FORMAT,
        "store": False,
    },
    region=REGION,
    attempts=1,
    timeout=TIMEOUT,
)
elapsed = time.perf_counter() - started
print(f"\nbedrock-runtime /openai/v1/responses {runtime_id}")
print(f"   HTTP {code} in {elapsed:.2f}s: {err(data)[:88]}")
if code == -1:
    print(f"=> no answer here either within {TIMEOUT}s, so today this endpoint draws")
    print("   no contrast with mantle. Re-run before relying on either.")
else:
    print(f"=> refused here, in words, in {elapsed:.2f}s, against the {TIMEOUT}s that")
    print("   bought nothing on mantle. Test structured output against the endpoint")
    print("   you will deploy on: here you get an error to handle, on mantle you get")
    print("   silence to time out.")

model                   HTTP  json.loads  lenient   keys  content[:34]
------------------------------------------------------------------------------------------------
openai.gpt-oss-120b      200      raises       ok   True  '**{   "summary":"The reviewer is l'
openai.gpt-oss-120b      200          ok       ok   True  '{"summary":"The reviewer loves the'
openai.gpt-oss-20b       200      raises       ok   True  '**Review{"summary": "Short review '
openai.gpt-oss-20b       200      raises       ok   True  '**Review Breakdown{"summary":"Batt'

=> answered: 4/4 (status -1 is this route not coming back within 45s, the same result as the table above)
=> of those, json.loads() raised on 3/4 and parse_json_lenient() recovered 4/4
=> schema keys exactly right after the lenient parse: 4/4
   So response_format here is accepted and loosely followed, not enforced.
   The schema's fields come back. Whether they arrive as a bare JSON object
   or wrapped in a prose heading that json.loads() reject

Structured output on `gpt-oss` therefore has no enforced route on `bedrock-mantle`:
the forced tool is accepted and ignored (§7 measures that), `text.format` does not
come back, and `response_format` on Chat Completions returns the fields wrapped in
whatever prose the model felt like adding. Ask for JSON, parse leniently and
validate — which is what §7 does — or use a hosted GPT model, where the table above
shows the constraint being honoured.

## 6. Strict JSON via a forced tool call

The portable alternative: define a tool whose *input schema is your output
schema*, then force it. This works on models that lack native structured output,
and the result arrives already parsed at the tool layer.

In [14]:
EMIT_TOOL = {
    "type": "function",
    "name": "emit_review",
    "description": "Return the structured review analysis.",
    "parameters": REVIEW_SCHEMA,
}


def emit_structured(
    text: str, model: str = SOL, attempts: int = 3, prefix: str = GPT5_PREFIX
) -> dict:
    """Forced-tool structured output, with a retry.

    Forcing tool_choice is honoured *almost* always. Production code should handle
    the occasional turn that returns prose instead, rather than indexing [0].
    """
    for attempt in range(attempts):
        code, data = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": f"Analyse this review:\n{text}",
                "tools": [EMIT_TOOL],
                "tool_choice": {"type": "function", "name": "emit_review"},
                "max_output_tokens": 700,
                "store": False,
            },
            region=REGION,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
        if calls:
            if attempt:
                print(f"(succeeded on attempt {attempt + 1})")
            return parse_json_lenient(calls[0]["arguments"])
        print(f"attempt {attempt + 1}: no tool call — retrying")
    raise RuntimeError("model would not emit the forced tool call")


print(json.dumps(emit_structured(REVIEW), indent=2))

{
  "summary": "The reviewer praises the superb battery life and fast charging, but criticizes the screen for scratching too easily and customer support for taking a week to respond.",
  "sentiment": "neutral",
  "topics": [
    "battery life",
    "charging speed",
    "screen durability",
    "customer support response time"
  ],
  "would_recommend": false
}


### Which route should you use?

| | `text.format` | forced tool |
|---|---|---|
| Availability | the hosted GPT line, GPT-6 included (§5b measures it per model); gemma-4 and grok, in their own notebooks | anywhere **constrained `tool_choice`** works |
| Result arrives as | string to parse | arguments (already structured) |
| Enum enforcement | yes | yes |
| Extra round trip | no | no |
| Fails when… | model lacks support; on gpt-oss that is silence, not a 400 (§5b) | model only supports `tool_choice:"auto"` (e.g. gpt-oss) |

There is a third route, used in §7 where neither of the above works: ask for
JSON in the prompt and parse leniently. It is the weakest of the three, because
it constrains nothing. §7 checks the result against the schema's enum to show
that difference rather than describe it.

Use `text.format` when the model supports it, and keep the forced-tool version as
your portable fallback. Which models support it is a per-model fact: read §5b's
table rather than inferring it from a version number.

## 7. The same, on gpt-oss (bare `/v1`)

Most of what is above works on the open-weight models with only the path
changed. Two things do not, and both are measured rather than described: the
forced-tool route below, and `text.format` in §5b.

In [15]:
result = run_tool_loop("Is B-200 in stock?", model=OSS120, prefix=OSS_PREFIX)
print("rounds:", result["rounds"])
for step in result["trace"]:
    print(f"   {step['tool']}({step['args']}) -> {step['result']}")
print("answer:", result["answer"][:200])

rounds: 2
   get_stock({'sku': 'B-200'}) -> {'sku': 'B-200', 'quantity': 0, 'in_stock': False}
answer: The SKU **B‑200** is currently **out of stock** (0 units available). Let me know if you’d like to be notified when it’s back, or if I can help you find an alternative product!


In [16]:
# On gpt-oss the forced-tool trick does NOT work: tool_choice only supports
# "auto", and the named form is accepted-but-ignored. Show that honestly.
try:
    print(
        json.dumps(
            emit_structured(REVIEW, model=OSS120, prefix=OSS_PREFIX, attempts=2),
            indent=2,
        )
    )
except RuntimeError as exc:
    print("forced tool on gpt-oss ->", exc)
    print("\nFallback for gpt-oss: ask for JSON in the prompt and parse leniently.")
    code, data = post(
        f"{OSS_PREFIX}/responses",
        {
            "model": OSS120,
            "input": (
                "Analyse this review and reply with ONLY a JSON object with keys "
                f"summary, sentiment, topics, would_recommend.\n{REVIEW}"
            ),
            "max_output_tokens": 700,
            "store": False,
        },
        region=REGION,
    )
    raw = response_text(data)
    print("HTTP", code, "| raw:", repr(raw[:120]))
    if raw.strip():
        parsed = parse_json_lenient(raw)
        print(json.dumps(parsed, indent=2))
        # The fallback buys well-formed JSON, NOT a honoured schema. Nothing
        # constrains the values, so check rather than assume: `sentiment` is
        # supposed to be one of three enum members.
        allowed = REVIEW_SCHEMA["properties"]["sentiment"]["enum"]
        got = parsed.get("sentiment")
        missing = [k for k in REVIEW_SCHEMA["required"] if k not in parsed]
        print(f"\nall required keys present : {not missing}"
              f"{'' if not missing else '  missing ' + str(missing)}")
        print(f"sentiment {got!r} in {allowed}: {got in allowed}")
        print("=> Prompting for JSON is not the same as enforcing a schema. Validate"
              "\n   the parsed object yourself before it reaches anything downstream.")

attempt 1: no tool call — retrying


attempt 2: no tool call — retrying
forced tool on gpt-oss -> model would not emit the forced tool call

Fallback for gpt-oss: ask for JSON in the prompt and parse leniently.


HTTP 200 | raw: '{\n  "summary": "The device has excellent battery life and fast charging, but the screen scratches easily and customer su'
{
  "summary": "The device has excellent battery life and fast charging, but the screen scratches easily and customer support is slow to respond.",
  "sentiment": "mixed",
  "topics": [
    "battery life",
    "charging speed",
    "screen durability",
    "customer support"
  ],
  "would_recommend": false
}

all required keys present : True
sentiment 'mixed' in ['positive', 'neutral', 'negative']: False
=> Prompting for JSON is not the same as enforcing a schema. Validate
   the parsed object yourself before it reaches anything downstream.


## 8. Failure modes worth handling

Three things that actually happen in production.

In [17]:
# (a) The model invents arguments that don't match your schema. Validate.
#
# FORCE the call. With tool_choice="auto" the model answers "I don't know which
# SKU" in prose and no arguments are invented at all, so the cell demonstrated
# nothing. Compelling the call is what surfaces the invented value.
code, data = post(
    f"{GPT5_PREFIX}/responses",
    {
        "model": SOL,
        "input": "Check stock for the blue widget please.",
        "tools": TOOLS,
        "tool_choice": {"type": "function", "name": "get_stock"},
        "max_output_tokens": 400,
        "store": False,
    },
    region=REGION,
)
calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
print("(a) invented arguments:")
if not calls:
    print("    no tool call even when forced — see §3, forcing is best-effort")
for call in calls:
    args = parse_json_lenient(call["arguments"] or "{}")
    known = str(args.get("sku", "")).upper() in INVENTORY
    print(f"    {call['name']}({args}) -> sku known to us? {known}")
print("    => always validate arguments against your own data before executing.")
print("       'the blue widget' is not a SKU, so whatever arrived above was made up.")

(a) invented arguments:
    get_stock({'sku': 'blue widget'}) -> sku known to us? False
    => always validate arguments against your own data before executing.
       'the blue widget' is not a SKU, so whatever arrived above was made up.


In [18]:
# (b) A tool that raises. Return a structured error rather than crashing the loop —
# the model can then apologise or try a different approach.
def risky_tool(sku: str) -> dict:
    if sku.upper() not in INVENTORY:
        raise KeyError(sku)
    return get_stock(sku)


def safe_call(name: str, args: dict) -> dict:
    try:
        return risky_tool(**args) if name == "get_stock" else DISPATCH[name](**args)
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}", "recoverable": True}


print("(b) tool error surfaced as data:")
print("   ", safe_call("get_stock", {"sku": "Z-999"}))

(b) tool error surfaced as data:
    {'error': "KeyError: 'Z-999'", 'recoverable': True}


In [19]:
# (c) A tight token budget can truncate before any JSON is emitted — HTTP 200 with
# an empty string. Check before parsing.
print("(c) budget truncation:")
for budget in (16, 700):
    code, data = post(
        f"{GPT5_PREFIX}/responses",
        {
            "model": SOL,
            "input": f"Analyse: {REVIEW}",
            "max_output_tokens": budget,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "review",
                    "schema": REVIEW_SCHEMA,
                    "strict": True,
                }
            },
            "store": False,
        },
        region=REGION,
    )
    text = response_text(data)
    status = data.get("status")
    print(
        f"    max_output_tokens={budget:4} HTTP {code} status={status!s:12} "
        f"chars={len(text)}"
    )
print("    => check for content before calling any parser")

(c) budget truncation:


    max_output_tokens=  16 HTTP 200 status=incomplete   chars=50


    max_output_tokens= 700 HTTP 200 status=completed    chars=246
    => check for content before calling any parser


## 9. An agent with validation and retries

Putting the pieces together.

In [20]:
class ToolAgent:
    """Client-side tool agent with validation, error surfacing and round limits."""

    def __init__(
        self, model=SOL, region=REGION, tools=None, dispatch=None, project=None
    ):
        self.model, self.region, self.project = model, region, project
        self.prefix = GPT5_PREFIX if model.startswith("openai.gpt-5.") else OSS_PREFIX
        self.tools = tools or TOOLS
        self.dispatch = dispatch or DISPATCH

    def _execute(self, name, args):
        fn = self.dispatch.get(name)
        if not fn:
            return {"error": f"unknown tool {name}"}
        try:
            return fn(**args)
        except Exception as exc:  # never let a tool kill the loop
            return {"error": f"{type(exc).__name__}: {exc}"}

    def run(self, question, max_rounds=6, max_output_tokens=600):
        conversation = [{"role": "user", "content": question}]
        headers = {"OpenAI-Project": self.project} if self.project else None
        for _ in range(max_rounds):
            code, data = post(
                f"{self.prefix}/responses",
                {
                    "model": self.model,
                    "input": conversation,
                    "tools": self.tools,
                    "tool_choice": "auto",
                    "max_output_tokens": max(16, max_output_tokens),
                    "store": False,
                },
                region=self.region,
                headers=headers,
            )
            if code != 200:
                raise RuntimeError(f"HTTP {code}: {err(data)}")
            calls = [
                i for i in data.get("output", []) if i.get("type") == "function_call"
            ]
            if not calls:
                return response_text(data)
            for call in calls:
                try:
                    args = parse_json_lenient(call["arguments"] or "{}")
                except ValueError:
                    args = {}
                output = self._execute(call["name"], args)
                conversation.append(
                    {
                        "type": "function_call",
                        "call_id": call["call_id"],
                        "name": call["name"],
                        "arguments": call["arguments"],
                    }
                )
                conversation.append(
                    {
                        "type": "function_call_output",
                        "call_id": call["call_id"],
                        "output": json.dumps(output),
                    }
                )
        return "(max rounds reached)"


agent = ToolAgent()
print(
    agent.run(
        "Which of A-100, B-200 and C-300 are in stock, and what is the "
        "cheapest shipping?"
    )[:400]
)

- **In stock:** A-100 (42 units), C-300 (7 units)
- **Out of stock:** B-200
- **Cheapest shipping:** Standard — **$4.99**


## Gotchas — tools and structured output

| Gotcha | Detail |
|---|---|
| Tool shape differs by API | Responses = flat; Chat Completions = nested under `function` |
| `call_id` | Must match between `function_call` and `function_call_output` |
| Reasoning replay | Never append reasoning items to the conversation |
| `tool_choice` support | **gpt-oss: `auto` only.** `none`/`required` 400; named form 200-but-ignored |
| Forced `tool_choice` | Best-effort even where supported — handle the prose turn |
| `json.loads` | Use lenient parsing; arguments can carry trailing characters |
| Invented arguments | Validate against your own data before executing |
| Tool exceptions | Return structured errors so the model can recover |
| Tight budgets | Truncation gives HTTP 200 + empty output — check before parsing |
| Parallel calls | Supported on gpt-5.6; Gemma 4 does one per turn |
| `max_tool_calls` | Accepted and **not enforced** (§4). Cap tool calls in your own loop |
| `text.format` on gpt-oss | Returned nothing on `bedrock-mantle`, not a 400 (§5b). Set your own timeout |
| `response_format` on gpt-oss | Answered 200 and followed the schema loosely; fields arrived wrapped in prose (§5b) |
| The failure differs by endpoint | What goes quiet on mantle comes back as a 404 on `bedrock-runtime`, timed in §5b |

## Next
- `04-prompt-caching-and-cost.ipynb` — cache the tool definitions above
- `05-server-side-tools-and-fine-tuning.ipynb` — let Bedrock run the tools
- `02-web-search-and-grounding.ipynb` — a server-side tool you don't implement